# Prebuilt middleware

LangChain và [Deep Agents](https://docs.langchain.com/oss/python/deepagents/overview) cung cấp các middleware được xây dựng sẵn cho các trường hợp sử dụng phổ biến. Mỗi middleware đều sẵn sàng cho môi trường production và có thể tùy chỉnh theo nhu cầu cụ thể của bạn.

## Middleware không phụ thuộc provider

Các middleware sau đây hoạt động với bất kỳ provider LLM nào:

| Middleware                                    | Mô tả                                                                                                  |
| --------------------------------------------- | ------------------------------------------------------------------------------------------------------ |
| [Tool error](https://docs.langchain.com/oss/python/langchain/middleware/built-in#tool-error)                     | Bắt các ngoại lệ khi thực thi tool và chuyển đổi chúng thành các thông báo lỗi cho model.             |
| [Tool retry](https://docs.langchain.com/oss/python/langchain/middleware/built-in#tool-retry)                     | Tự động thử lại các lệnh gọi tool thất bại với cơ chế exponential backoff (độ trễ tăng dần theo hàm mũ). |
| [Model retry](https://docs.langchain.com/oss/python/langchain/middleware/built-in#model-retry)                   | Tự động thử lại các lệnh gọi model thất bại với cơ chế exponential backoff.                              |
| [Model fallback](https://docs.langchain.com/oss/python/langchain/middleware/built-in#model-fallback)             | Tự động chuyển sang các model thay thế khi model chính gặp lỗi.                              |
| [Summarization](https://docs.langchain.com/oss/python/langchain/middleware/built-in#summarization)               | Tự động tóm tắt lịch sử trò chuyện khi sắp đạt đến giới hạn token.                   |
| [Human-in-the-loop](https://docs.langchain.com/oss/python/langchain/middleware/built-in#human-in-the-loop)       | Tạm dừng quá trình thực thi để con người phê duyệt các lệnh gọi tool.                                             |
| [Model call limit](https://docs.langchain.com/oss/python/langchain/middleware/built-in#model-call-limit)         | Giới hạn số lượng lệnh gọi model để ngăn chặn chi phí phát sinh quá mức.                                   |
| [Tool call limit](https://docs.langchain.com/oss/python/langchain/middleware/built-in#tool-call-limit)           | Kiểm soát việc thực thi tool bằng cách giới hạn số lượng lệnh gọi.                                |
| [PII detection](https://docs.langchain.com/oss/python/langchain/middleware/built-in#pii-detection)               | Phát hiện và xử lý thông tin định danh cá nhân (PII).                                  |
| [To-do list](https://docs.langchain.com/oss/python/langchain/middleware/built-in#to-do-list)                     | Trang bị cho agent khả năng lập kế hoạch và theo dõi tiến độ công việc.                                   |
| [LLM tool selector](https://docs.langchain.com/oss/python/langchain/middleware/built-in#llm-tool-selector)       | Sử dụng một LLM để chọn các tool liên quan trước khi gọi model chính.                                |
| [Provider tool search](https://docs.langchain.com/oss/python/langchain/middleware/built-in#provider-tool-search) | Trì hoãn các tool đằng sau tính năng tìm kiếm tool trên server của provider, chỉ hiển thị chúng khi cần thiết.              |
| [Shell tool](https://docs.langchain.com/oss/python/langchain/middleware/built-in#shell-tool)                     | Cung cấp một phiên shell duy trì cho agent để thực thi lệnh.                            |
| [Filesystem](https://docs.langchain.com/oss/python/langchain/middleware/built-in#filesystem-middleware)          | Cung cấp cho agent một hệ thống tệp để lưu trữ context và bộ nhớ dài hạn.                  |
| [Subagent](https://docs.langchain.com/oss/python/langchain/middleware/built-in#subagent)                         | Bổ sung khả năng sinh ra các subagent.                                                           |
| [Rubric grading (Beta)](https://docs.langchain.com/oss/python/langchain/middleware/built-in#rubric-grading)      | Áp dụng phương pháp LLM-as-a-judge để agent tự đánh giá và lặp lại quá trình cho đến khi thỏa mãn một rubric. |
| [File search](https://docs.langchain.com/oss/python/langchain/middleware/built-in#file-search)                   | Cung cấp các công cụ tìm kiếm Glob và Grep trên các tệp của hệ thống tệp.                                     |
| [Context editing](https://docs.langchain.com/oss/python/langchain/middleware/built-in#context-editing)           | Quản lý context trò chuyện bằng cách cắt bớt hoặc xóa bỏ các kết quả sử dụng tool.                                |
| [LLM tool emulator](https://docs.langchain.com/oss/python/langchain/middleware/built-in#llm-tool-emulator)       | Giả lập quá trình thực thi tool bằng cách sử dụng một LLM cho mục đích kiểm thử.                                |

### Tool error

Bắt các ngoại lệ phát sinh trong quá trình thực thi tool và chuyển đổi chúng thành các `ToolMessage` báo lỗi mà model có thể nhìn thấy và phục hồi, thay vì làm dừng đột ngột quá trình chạy của agent. Tool error rất hữu ích cho các trường hợp sau:

* Cho phép model thử gọi lại một tool đã thất bại với các tham số được sửa đổi.
* Hiển thị các thông báo lỗi đã được kiểm soát, làm sạch thay vì chi tiết ngoại lệ thô.
* Ngăn chặn các ngoại lệ tool không lường trước làm hỏng agent.

<div class="alert alert-info">

Middleware Tool error không tự động thử lại các lệnh gọi bị lỗi. Để thực hiện thử lại, hãy kết hợp với middleware [Tool retry](https://docs.langchain.com/oss/python/langchain/middleware/built-in#tool-retry) và đặt nó ở vị trí *bên trong* (phía trước trong danh sách `middleware`), đồng thời cấu hình `on_failure="error"` để các ngoại lệ có thể truyền đến middleware Tool error. Xem [ví dụ đầy đủ](https://docs.langchain.com/oss/python/langchain/middleware/built-in#tool-error-full-example) bên dưới.

</div>

**Tài liệu API:** [`ToolErrorMiddleware`](https://reference.langchain.com/python/langchain/agents/middleware/tool_error/ToolErrorMiddleware)

<div class="alert alert-info">

`ToolErrorMiddleware` yêu cầu phiên bản `langchain>=1.3.14`.

</div>

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolErrorMiddleware


def on_error(exc: Exception, request: ToolCallRequest) -> str | None:
    if isinstance(exc, ValueError):
        return f"`{request.tool_call['name']}` thất bại với lỗi {type(exc).__name__}."
    # truyền tiếp tất cả các lỗi khác


agent = create_agent(
    model="gpt-5.5",
    tools=[your_tools],
    middleware=[ToolErrorMiddleware(on_error)],
)

**Các tùy chọn cấu hình**

* `on_error` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">Callable[[Exception, ToolCallRequest], str | list[ContentBlock] | None]</span>

  Trình xử lý đồng bộ được gọi cho mỗi ngoại lệ phát sinh trong quá trình thực thi tool. Trả về nội dung (một `str` hoặc danh sách các block nội dung) để chuyển đổi ngoại lệ thành một `ToolMessage(status="error")`. Trả về `None` hoặc bỏ qua câu lệnh return để cho phép ngoại lệ tiếp tục truyền đi. Được sử dụng trên luồng đồng bộ và luồng bất đồng bộ (trừ khi `aon_error` được cung cấp).

* `aon_error` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">Callable[[Exception, ToolCallRequest], Awaitable[str | list[ContentBlock] | None]]</span>
  
  Trình xử lý bất đồng bộ tùy chọn, được sử dụng trên luồng thực thi bất đồng bộ. Nếu không được cung cấp, hệ thống sẽ sử dụng fallback về `on_error`.

* `tools` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">list[BaseTool | str]</span>

  Danh sách tùy chọn gồm các tool hoặc tên tool để áp dụng xử lý lỗi. Nếu truyền `None`, sẽ áp dụng cho tất cả các tool.

**Ví dụ đầy đủ**

Trình xử lý `on_error` nhận ngoại lệ và `ToolCallRequest` (bao gồm dict của lệnh gọi tool chứa tên, argument và call ID). Trả về `None` đối với các ngoại lệ bạn không muốn xử lý, và chúng sẽ tiếp tục truyền đi như bình thường.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolErrorMiddleware, ToolRetryMiddleware


def on_error(exc: Exception, request: ToolCallRequest) -> str | None:
    # Hiển thị ValueError cho model để nó có thể sửa lại đầu vào
    if isinstance(exc, ValueError):
        return f"`{request.tool_call['name']}` thất bại: {type(exc).__name__}. Hãy sửa lại đầu vào và thử lại."
    # Cho phép truyền tiếp tất cả các ngoại lệ khác (dừng quá trình chạy)
    return None


# Cách dùng chỉ dành cho async
async def aon_error(exc: Exception, request: ToolCallRequest) -> str | None:
    if isinstance(exc, ConnectionError):
        return f"Tool `{request.tool_call['name']}` gặp lỗi kết nối."
    return None


agent = create_agent(
    model="gpt-5.5",
    tools=[search_tool, database_tool],
    middleware=[
        # Đặt retry ở bên trong để các ngoại lệ có thể đến được ToolErrorMiddleware sau khi đã thử lại hết số lần
        ToolRetryMiddleware(max_retries=3, on_failure="error"),
        ToolErrorMiddleware(on_error=on_error, tools=["search_tool"]),
    ],
)

# Chỉ async: chỉ truyền aon_error (không truyền on_error)
async_agent = create_agent(
    model="gpt-5.5",
    tools=[api_tool],
    middleware=[ToolErrorMiddleware(aon_error=aon_error)],
)

<div class="alert alert-info">

Ưu tiên trả về nội dung có nêu tên loại ngoại lệ thay vì tin nhắn ngoại lệ thô, vì tin nhắn thô có thể mang thông tin nhạy cảm hoặc chi tiết hệ thống nội bộ. Trình xử lý `on_error` kiểm soát việc tiết lộ thông tin: tin nhắn ngoại lệ thô sẽ không bao giờ được gửi đến model trừ khi bạn chủ động thêm nó vào.

</div>

### Tool retry

Tự động thử lại các lệnh gọi tool thất bại với cơ chế exponential backoff (độ trễ tăng theo hàm mũ) có thể cấu hình. Tool retry rất hữu ích cho các trường hợp sau:

* Xử lý các lỗi tạm thời khi gọi API bên ngoài.
* Cải thiện độ tin cậy của các tool phụ thuộc vào mạng.
* Xây dựng các agent bền bỉ có khả năng xử lý mượt mà các lỗi tạm thời.

**Tài liệu API:** [`ToolRetryMiddleware`](https://reference.langchain.com/python/langchain/agents/middleware/tool_retry/ToolRetryMiddleware)

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolRetryMiddleware

agent = create_agent(
    model="gpt-5.5",
    tools=[search_tool, database_tool],
    middleware=[
        ToolRetryMiddleware(
            max_retries=3,
            backoff_factor=2.0,
            initial_delay=1.0,
        ),
    ],
)

**Các tùy chọn cấu hình**

* `max_retries` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">number</span> <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">default:"2"</span>
  
  Số lần thử lại tối đa sau lệnh gọi ban đầu (tổng cộng 3 lần thử nếu để mặc định).

* `tools` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">list[BaseTool | str]</span>
  
  Danh sách tùy chọn các tool hoặc tên tool cần áp dụng logic thử lại. Nếu truyền `None`, sẽ áp dụng cho tất cả các tool.

* `retry_on` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">tuple[type[Exception], ...] | callable</span> <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">default:"default_retry_on"</span>
  
  Có thể là một tuple chứa các loại ngoại lệ để tiến hành thử lại, hoặc một hàm callable nhận vào một ngoại lệ và trả về `True` nếu cần thử lại. Từ phiên bản `langchain>=1.3.16`, giá trị mặc định sẽ thử lại đối với các [lỗi model có thể thử lại](https://docs.langchain.com/oss/python/langchain/models#model-exceptions) và tất cả các ngoại lệ chưa được phân loại, đồng thời không còn thử lại các lỗi model được đánh dấu là không thể thử lại.

* `on_failure` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">string | callable</span> <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">default:"continue"</span>
  
  Hành vi khi đã dùng hết số lần thử lại. Các tùy chọn:

  * `'continue'` (mặc định) - Trả về một `ToolMessage` chứa chi tiết lỗi, cho phép LLM tự xử lý sự cố.
  * `'error'` - Ném lại ngoại lệ, làm ngừng quá trình thực thi của agent.
  * Custom callable - Hàm nhận vào một ngoại lệ và trả về một chuỗi string dùng làm nội dung cho `ToolMessage`.

* `backoff_factor` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">number</span> <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">default:"2.0"</span>

  Hệ số nhân cho exponential backoff. Mỗi lần thử lại sẽ chờ `initial_delay * (backoff_factor ** retry_number)` giây. Đặt thành `0.0` để duy trì độ trễ cố định.

* `initial_delay` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">number</span> <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">default:"1.0"</span>
  
  Độ trễ ban đầu tính bằng giây trước lần thử lại đầu tiên.

* `max_delay` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">number</span> <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">default:"60.0"</span>
  
  Độ trễ tối đa tính bằng giây giữa các lần thử lại (giới hạn mức tăng của exponential backoff).

* `jitter` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">boolean</span> <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">default:"true"</span>
  
  Xác định có thêm biến động ngẫu nhiên (jitter) khoảng `±25%` vào độ trễ để tránh hiện tượng thundering herd hay không.

**Ví dụ đầy đủ**

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolRetryMiddleware


agent = create_agent(
    model="gpt-5.5",
    tools=[search_tool, database_tool, api_tool],
    middleware=[
        ToolRetryMiddleware(
            max_retries=3,
            backoff_factor=2.0,
            initial_delay=1.0,
            max_delay=60.0,
            jitter=True,
            tools=["api_tool"],
            retry_on=(ConnectionError, TimeoutError),
            on_failure="continue",
        ),
    ],
)

### Model retry

Tự động thử lại các lệnh gọi model thất bại với cơ chế exponential backoff có thể cấu hình. Model retry rất hữu ích cho các trường hợp sau:

* Xử lý các lỗi tạm thời trong các lệnh gọi API của model.
* Cải thiện độ tin cậy của các yêu cầu model phụ thuộc vào mạng.
* Xây dựng các agent bền bỉ có khả năng xử lý mượt mà các lỗi model tạm thời.

**Tài liệu API:** [`ModelRetryMiddleware`](https://reference.langchain.com/python/langchain/agents/middleware/model_retry/ModelRetryMiddleware)

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelRetryMiddleware

agent = create_agent(
    model="gpt-5.5",
    tools=[search_tool, database_tool],
    middleware=[
        ModelRetryMiddleware(
            max_retries=3,
            backoff_factor=2.0,
            initial_delay=1.0,
        ),
    ],
)

**Các tùy chọn cấu hình**
  
* `max_retries` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">number</span> <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">default:"2"</span>
  
  Số lần thử lại tối đa sau lệnh gọi ban đầu (tổng cộng 3 lần thử nếu để mặc định).

* `retry_on` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">tuple[type[Exception], ...] | callable</span> <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">default:"default_retry_on"</span>

  Có thể là một tuple chứa các loại ngoại lệ để tiến hành thử lại, hoặc một hàm callable nhận vào một ngoại lệ và trả về `True` nếu cần thử lại. Từ phiên bản `langchain>=1.3.16`, giá trị mặc định sẽ thử lại đối với các [lỗi model có thể thử lại](https://docs.langchain.com/oss/python/langchain/models#model-exceptions) và tất cả các ngoại lệ chưa được phân loại, đồng thời không còn thử lại các lỗi model được đánh dấu là không thể thử lại.

* `on_failure` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">string | callable</span> <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">default:"continue"</span>
    
  Hành vi khi đã dùng hết số lần thử lại. Các tùy chọn:

  * `'continue'` (mặc định) - Trả về một `AIMessage` chứa chi tiết lỗi, cho phép agent có khả năng xử lý tình huống một cách mượt mà.
  * `'error'` - Ném lại ngoại lệ (làm dừng quá trình thực thi của agent).
  * Custom callable - Hàm nhận vào một ngoại lệ và trả về một chuỗi string dùng làm nội dung cho `AIMessage`.

* `backoff_factor` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">number</span> <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">default:"2.0"</span>

  Hệ số nhân cho exponential backoff. Mỗi lần thử lại sẽ chờ `initial_delay * (backoff_factor ** retry_number)` giây. Đặt thành `0.0` để duy trì độ trễ cố định.

* `initial_delay` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">number</span> <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">default:"1.0"</span>
  
  Độ trễ ban đầu tính bằng giây trước lần thử lại đầu tiên.

* `max_delay` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">number</span> <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">default:"60.0"</span>
  
  Độ trễ tối đa tính bằng giây giữa các lần thử lại (giới hạn mức tăng của exponential backoff).

* `jitter` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">boolean</span> <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">default:"true"</span>
  
  Xác định có thêm biến động ngẫu nhiên (jitter) khoảng `±25%` vào độ trễ để tránh hiện tượng thundering herd hay không.

**Ví dụ đầy đủ**

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelRetryMiddleware


# Cách dùng cơ bản với cài đặt mặc định (2 lần thử lại, exponential backoff)
agent = create_agent(
    model="gpt-5.5",
    tools=[search_tool],
    middleware=[ModelRetryMiddleware()],
)

# Lọc ngoại lệ tùy chỉnh
class TimeoutError(Exception):
    """Ngoại lệ tùy chỉnh cho các lỗi timeout."""
    pass

class ConnectionError(Exception):
    """Ngoại lệ tùy chỉnh cho các lỗi kết nối."""
    pass

# Chỉ thử lại với các ngoại lệ cụ thể
retry = ModelRetryMiddleware(
    max_retries=4,
    retry_on=(TimeoutError, ConnectionError),
    backoff_factor=1.5,
)


def should_retry(error: Exception) -> bool:
    # Chỉ thử lại khi gặp lỗi giới hạn tần suất (rate limit)
    if isinstance(error, TimeoutError):
        return True
    # Hoặc kiểm tra các mã trạng thái HTTP cụ thể
    if hasattr(error, "status_code"):
        return error.status_code in (429, 503)
    return False

retry_with_filter = ModelRetryMiddleware(
    max_retries=3,
    retry_on=should_retry,
)

# Trả về thông báo lỗi thay vì ném ra ngoại lệ
retry_continue = ModelRetryMiddleware(
    max_retries=4,
    on_failure="continue",  # Trả về AIMessage chứa lỗi thay vì ném ra ngoại lệ
)

# Định dạng thông báo lỗi tùy chỉnh
def format_error(error: Exception) -> str:
    return f"Lệnh gọi model thất bại: {error}. Vui lòng thử lại sau."

retry_with_formatter = ModelRetryMiddleware(
    max_retries=4,
    on_failure=format_error,
)

# Độ trễ cố định (không tăng theo hàm mũ)
constant_backoff = ModelRetryMiddleware(
    max_retries=5,
    backoff_factor=0.0,  # Không tăng theo hàm mũ
    initial_delay=2.0,  # Luôn đợi 2 giây
)

# Ném ra ngoại lệ khi thất bại
strict_retry = ModelRetryMiddleware(
    max_retries=2,
    on_failure="error",  # Ném lại ngoại lệ thay vì trả về thông báo
)

### Model fallback

Tự động chuyển sang các model thay thế khi model chính thất bại. Model fallback rất hữu ích cho các trường hợp sau:

* Xây dựng các agent đáng tin cậy có khả năng xử lý khi model gặp sự cố gián đoạn.
* Tối ưu hóa chi phí bằng cách chuyển sang các model rẻ hơn.
* Dự phòng provider giữa OpenAI, Anthropic, v.v.

**Tài liệu API:** [`ModelFallbackMiddleware`](https://reference.langchain.com/python/langchain/agents/middleware/model_fallback/ModelFallbackMiddleware)

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelFallbackMiddleware

agent = create_agent(
    model="gpt-5.5",
    tools=[],
    middleware=[
        ModelFallbackMiddleware(
            "gpt-5.4-mini",
            "claude-3-5-sonnet-20241022",
        ),
    ],
)

<div class="alert alert-info">

Xem [video hướng dẫn](https://www.youtube.com/watch?v=8rCRO0DUeIM) minh họa hành vi của middleware model fallback.

</div>

**Các tùy chọn cấu hình**
  
* `first_model` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">string | BaseChatModel</span> <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">required</span>
    
  Model dự phòng đầu tiên sẽ được thử khi model chính thất bại. Có thể là một chuỗi định danh model (ví dụ: `'openai:gpt-5.4-mini'`) hoặc một instance của `BaseChatModel`.

* `*additional_models` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">string | BaseChatModel</span>

  Các model dự phòng bổ sung sẽ được thử nghiệm theo thứ tự nếu các model trước đó thất bại.

### Summarization

Tự động tóm tắt lịch sử trò chuyện khi sắp đạt đến giới hạn token, giúp bảo tồn các tin nhắn gần đây trong khi nén context cũ hơn. Summarization rất hữu ích cho các trường hợp sau:

* Các cuộc trò chuyện kéo dài vượt quá context window.
* Các đoạn hội thoại nhiều lượt với lịch sử dài.
* Các ứng dụng yêu cầu bảo tồn toàn bộ context của cuộc trò chuyện.

<div class="alert alert-info">

Tóm tắt là hình thức nén context hướng tới văn bản. Nó không thực hiện thay đổi kích thước, lấy mẫu xuống hoặc nén các tải trọng hình ảnh/âm thanh/video. Các tin nhắn gần đây được giữ lại bởi điều kiện `keep` vẫn bao gồm các block đa phương thức gốc, trong khi các tin nhắn đa phương thức cũ hơn bị tóm tắt sẽ chỉ được biểu diễn bằng văn bản tóm tắt được tạo ra. Đối với các ứng dụng nặng về hình ảnh, hãy lưu trữ phương tiện trong filesystem hoặc object store, sau đó truyền URL hoặc tham chiếu file thông qua lịch sử tin nhắn.

</div>

**Tài liệu API:** [`SummarizationMiddleware`](https://reference.langchain.com/python/langchain/agents/middleware/summarization/SummarizationMiddleware)

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="gpt-5.5",
    tools=[your_weather_tool, your_calculator_tool],
    middleware=[
        SummarizationMiddleware(
            model="gpt-5.4-mini",
            trigger=("tokens", 4000),
            keep=("messages", 20),
        ),
    ],
)

**Các tùy chọn cấu hình**

<div class="alert alert-success">

Các điều kiện `fraction` (tỷ lệ phần trăm) dành cho `trigger` và `keep` (hiển thị bên dưới) dựa vào [dữ liệu profile](https://docs.langchain.com/oss/python/langchain/models#model-profiles) của chat model nếu bạn đang dùng `langchain>=1.1`. Nếu không có sẵn dữ liệu, hãy dùng điều kiện khác hoặc chỉ định thủ công:

```python
from langchain.chat_models import init_chat_model

custom_profile = {
    "max_input_tokens": 100_000,
    # ...
}
model = init_chat_model("gpt-5.5", profile=custom_profile)
```

</div>

* `model` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">string | BaseChatModel</span> <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">required</span>

  Model để tạo các bản tóm tắt. Có thể là một chuỗi định danh model (ví dụ: `'openai:gpt-5.4-mini'`) hoặc một instance của `BaseChatModel`. Xem [`init_chat_model`](https://reference.langchain.com/python/langchain/chat_models/base/init_chat_model) để biết thêm thông tin.

* `trigger` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">ContextSize | TriggerClause | list[ContextSize | TriggerClause] | None</span>

  (Các) điều kiện kích hoạt việc tóm tắt. Có thể là:

  * Một tuple [`ContextSize`](https://reference.langchain.com/python/langchain/agents/middleware/summarization/ContextSize) (ngưỡng được chỉ định phải thỏa mãn)
  * Một dict [`TriggerClause`](https://reference.langchain.com/python/langchain/agents/middleware/summarization/TriggerClause) (tất cả các ngưỡng được chỉ định đều phải thỏa mãn - logic AND)
  * Một danh sách kết hợp bất kỳ định dạng nào (chỉ cần một item bất kỳ thỏa mãn - logic OR)

  Các ngưỡng được hỗ trợ:

  * `fraction` (float): Tỷ lệ phần trăm kích thước context của model (từ 0 đến 1)
  * `tokens` (int): Số lượng token tuyệt đối
  * `messages` (int): Số lượng tin nhắn

  Một tuple [`ContextSize`](https://reference.langchain.com/python/langchain/agents/middleware/summarization/ContextSize) thể hiện chính xác một ngưỡng. Một dict [`TriggerClause`](https://reference.langchain.com/python/langchain/agents/middleware/summarization/TriggerClause) có thể bao gồm một hoặc nhiều ngưỡng, ví dụ: `{"tokens": 4000, "messages": 10}`, và tất cả các ngưỡng trong dict phải được thỏa mãn (AND).

  Mỗi dict [`TriggerClause`](https://reference.langchain.com/python/langchain/agents/middleware/summarization/TriggerClause) phải chỉ định ít nhất một ngưỡng. Nếu `trigger` không được cung cấp, việc tóm tắt sẽ không được kích hoạt tự động.

  Xem tài liệu API về [`ContextSize`](https://reference.langchain.com/python/langchain/agents/middleware/summarization/ContextSize) và [`TriggerClause`](https://reference.langchain.com/python/langchain/agents/middleware/summarization/TriggerClause) để biết thêm thông tin.

* `keep` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">ContextSize</span> <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">default:"('messages', 20)"</span>
  
  Lượng context được giữ lại sau khi tóm tắt. Phải chỉ định chính xác một trong các tùy chọn:

  * `fraction` (float): Tỷ lệ phần trăm kích thước context của model cần giữ lại (từ 0 đến 1)
  * `tokens` (int): Số lượng token tuyệt đối cần giữ lại
  * `messages` (int): Số lượng tin nhắn gần đây cần giữ lại

  Xem tài liệu API về [`ContextSize`](https://reference.langchain.com/python/langchain/agents/middleware/summarization/ContextSize) để biết thêm thông tin.

* `token_counter` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">function</span>

  Hàm đếm token tùy chỉnh. Mặc định là đếm dựa trên ký tự.

* `summary_prompt` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">string</span>
  
  Mẫu prompt tùy chỉnh dành cho việc tóm tắt. Sử dụng mẫu tích hợp sẵn nếu không được chỉ định. Mẫu nên bao gồm placeholder `{messages}` - vị trí mà lịch sử trò chuyện sẽ được chèn vào.

* `trim_tokens_to_summarize` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">number</span> <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">default:"4000"</span>

  Số lượng token tối đa được đưa vào khi tạo bản tóm tắt. Các tin nhắn sẽ được cắt xén cho vừa giới hạn này trước khi được tóm tắt.

**Ví dụ đầy đủ**

Middleware summarization theo dõi số lượng token của tin nhắn và tự động tóm tắt các tin nhắn cũ hơn khi đạt đến các ngưỡng nhất định.

**Các điều kiện trigger** kiểm soát khi nào việc tóm tắt sẽ chạy:

* Một ngưỡng duy nhất: kích hoạt khi ngưỡng đó được thỏa mãn
* Một mệnh đề trigger với nhiều ngưỡng: chỉ kích hoạt khi tất cả các ngưỡng được thỏa mãn (logic AND)
* Một danh sách các điều kiện trigger: kích hoạt khi bất kỳ item nào được thỏa mãn (logic OR)
* Mỗi ngưỡng có thể sử dụng `fraction` (dựa trên kích thước context của model), `tokens` (số lượng tuyệt đối) hoặc `messages` (số lượng tin nhắn).

**Điều kiện keep** kiểm soát lượng context sẽ được giữ lại (chỉ định chính xác một tùy chọn):

* `fraction` - Tỷ lệ phần trăm kích thước context của model cần giữ
* `tokens` - Số lượng token tuyệt đối cần giữ
* `messages` - Số lượng tin nhắn gần đây cần giữ

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware


# Một điều kiện: kích hoạt nếu số token >= 4000
agent = create_agent(
    model="gpt-5.5",
    tools=[your_weather_tool, your_calculator_tool],
    middleware=[
        SummarizationMiddleware(
            model="gpt-5.4-mini",
            trigger=("tokens", 4000),
            keep=("messages", 20),
        ),
    ],
)

# Nhiều điều kiện: kích hoạt nếu số token >= 3000 HOẶC số tin nhắn >= 6
agent2 = create_agent(
    model="gpt-5.5",
    tools=[your_weather_tool, your_calculator_tool],
    middleware=[
        SummarizationMiddleware(
            model="gpt-5.4-mini",
            trigger=[
                ("tokens", 3000),
                ("messages", 6),
            ],
            keep=("messages", 20),
        ),
    ],
)

# Logic AND: chỉ kích hoạt khi số token >= 4000 VÀ số tin nhắn >= 10
agent3 = create_agent(
    model="gpt-5.5",
    tools=[your_weather_tool, your_calculator_tool],
    middleware=[
        SummarizationMiddleware(
            model="gpt-5.4-mini",
            trigger={"tokens": 4000, "messages": 10},
            keep=("messages", 20),
        ),
    ],
)

# Kết hợp AND và OR: kích hoạt nếu (token >= 5000 VÀ tin nhắn >= 3)
# HOẶC (token >= 3000 VÀ tin nhắn >= 6)
agent4 = create_agent(
    model="gpt-5.5",
    tools=[your_weather_tool, your_calculator_tool],
    middleware=[
        SummarizationMiddleware(
            model="gpt-5.4-mini",
            trigger=[
                {"tokens": 5000, "messages": 3},
                {"tokens": 3000, "messages": 6},
            ],
            keep=("messages", 20),
        ),
    ],
)

# Sử dụng giới hạn theo tỷ lệ phần trăm
agent5 = create_agent(
    model="gpt-5.5",
    tools=[your_weather_tool, your_calculator_tool],
    middleware=[
        SummarizationMiddleware(
            model="gpt-5.4-mini",
            trigger=("fraction", 0.8),
            keep=("fraction", 0.3),
        ),
    ],
)

### Human-in-the-loop

Tạm dừng quá trình thực thi của agent để con người phê duyệt, chỉnh sửa hoặc từ chối các lệnh gọi tool trước khi chúng được chạy. [Human-in-the-loop](https://docs.langchain.com/oss/python/langchain/human-in-the-loop) rất hữu ích cho các trường hợp sau:

* Các thao tác có rủi ro cao đòi hỏi sự phê duyệt của con người (ví dụ: ghi vào database, giao dịch tài chính).
* Các quy trình tuân thủ mà sự giám sát của con người là bắt buộc.
* Các cuộc trò chuyện dài kỳ nơi phản hồi của con người sẽ dẫn dắt agent.

**Tài liệu API:** [`HumanInTheLoopMiddleware`](https://reference.langchain.com/python/langchain/agents/middleware/human_in_the_loop/HumanInTheLoopMiddleware)

<div class="alert alert-warning">

Middleware human-in-the-loop yêu cầu một [checkpointer](https://docs.langchain.com/oss/python/langgraph/checkpointers#checkpoints) để duy trì state thông qua các lần bị gián đoạn.

</div>

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def your_read_email_tool(email_id: str) -> str:
    """Hàm giả lập (mock) để đọc email theo ID."""
    return f"Nội dung email cho ID: {email_id}"

def your_send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Hàm giả lập (mock) để gửi email."""
    return f"Đã gửi email tới {recipient} với tiêu đề '{subject}'"

agent = create_agent(
    model="gpt-5.5",
    tools=[your_read_email_tool, your_send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "your_send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "your_read_email_tool": False,
            }
        ),
    ],
)

<div class="alert alert-success">

Để xem các ví dụ đầy đủ, các tùy chọn cấu hình và các pattern tích hợp, vui lòng tham khảo [Tài liệu human-in-the-loop](https://docs.langchain.com/oss/python/langchain/human-in-the-loop).

</div>

<div class="alert alert-info">

Xem [video hướng dẫn](https://www.youtube.com/watch?v=SpfT6-YAVPk) minh họa hành vi của middleware human-in-the-loop.

</div>

### Model call limit

Giới hạn số lần gọi model để ngăn chặn các vòng lặp vô hạn hoặc phát sinh chi phí quá mức. Giới hạn gọi model rất hữu ích cho các trường hợp sau:

* Ngăn chặn các agent bị lỗi thực hiện quá nhiều lệnh gọi API.
* Thiết lập kiểm soát chi phí trên các môi trường production.
* Kiểm thử hành vi của agent trong một ngân sách gọi API nhất định.

**Tài liệu API:** [`ModelCallLimitMiddleware`](https://reference.langchain.com/python/langchain/agents/middleware/model_call_limit/ModelCallLimitMiddleware)

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="gpt-5.5",
    checkpointer=InMemorySaver(),  # Bắt buộc để giới hạn theo thread
    tools=[],
    middleware=[
        ModelCallLimitMiddleware(
            thread_limit=10,
            run_limit=5,
            exit_behavior="end",
        ),
    ],
)

<div class="alert alert-info">

Xem [video hướng dẫn](https://www.youtube.com/watch?v=nJEER0uaNkE) minh họa hành vi của middleware Model Call Limit.

</div>

**Các tùy chọn cấu hình**

* `thread_limit` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">number</span>

  Số lệnh gọi model tối đa trên tất cả các lần chạy trong một thread. Mặc định là không giới hạn.

* `run_limit` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">number</span>
  
  Số lệnh gọi model tối đa cho mỗi lần được gọi. Mặc định là không giới hạn.

* `exit_behavior` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">string</span> <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">default:"end"</span>
  
  Hành vi khi đạt đến giới hạn. Tùy chọn: `'end'` (kết thúc mượt mà) hoặc `'error'` (ném ra ngoại lệ).

### Tool call limit

Kiểm soát quá trình thực thi của agent bằng cách giới hạn số lượng lệnh gọi tool, có thể áp dụng toàn cục cho tất cả các tool hoặc cho các tool cụ thể. Tool call limit rất hữu ích cho các trường hợp sau:

* Ngăn chặn việc gọi quá mức đến các API bên ngoài tốn kém chi phí.
* Giới hạn việc tìm kiếm web hoặc truy vấn database.
* Áp đặt rate limit lên một số tool nhất định.
* Bảo vệ hệ thống khỏi các vòng lặp agent vượt khỏi tầm kiểm soát.

**Tài liệu API:** [`ToolCallLimitMiddleware`](https://reference.langchain.com/python/langchain/agents/middleware/tool_call_limit/ToolCallLimitMiddleware)

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolCallLimitMiddleware

agent = create_agent(
    model="gpt-5.5",
    tools=[search_tool, database_tool],
    middleware=[
        # Giới hạn toàn cục
        ToolCallLimitMiddleware(thread_limit=20, run_limit=10),
        # Giới hạn cho từng tool cụ thể
        ToolCallLimitMiddleware(
            tool_name="search",
            thread_limit=5,
            run_limit=3,
        ),
    ],
)

<div class="alert alert-info">

Xem [video hướng dẫn](https://www.youtube.com/watch?v=6gYlaJJ8t0w) minh họa hành vi của middleware Tool Call Limit.

</div>

**Các tùy chọn cấu hình**

* `tool_name` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">string</span>

  Tên của tool cụ thể cần giới hạn. Nếu không cung cấp, các giới hạn sẽ áp dụng cho **tất cả các tool toàn cục**.

* `thread_limit` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">number</span>

  Số lệnh gọi tool tối đa trên tất cả các lần chạy trong một thread (cuộc trò chuyện). Sẽ được duy trì liên tục qua nhiều lần gọi với cùng một ID thread. Yêu cầu một checkpointer để duy trì state. Nếu truyền `None` có nghĩa là không giới hạn theo thread.

* `run_limit` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">number</span>

  Số lệnh gọi tool tối đa cho mỗi lần được gọi (một chu kỳ từ tin nhắn người dùng → phản hồi). Giá trị này sẽ được reset lại cho mỗi tin nhắn người dùng mới. Nếu truyền `None` có nghĩa là không giới hạn theo run.

  **Lưu ý:** Phải chỉ định ít nhất một trong hai thông số `thread_limit` hoặc `run_limit`.


* `exit_behavior` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">string</span> <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">default:"continue"</span>
  
  Hành vi khi đạt đến giới hạn:

  * `'continue'` (mặc định) - Chặn các lệnh gọi tool vượt quá giới hạn bằng thông báo lỗi, cho phép các tool khác và model tiếp tục hoạt động. Model sẽ quyết định khi nào nên kết thúc dựa trên thông báo lỗi.
  * `'error'` - Ném ra ngoại lệ `ToolCallLimitExceededError`, làm ngừng thực thi ngay lập tức.
  * `'end'` - Ngừng thực thi ngay lập tức với một `ToolMessage` và tin nhắn AI dành cho lệnh gọi tool đã vượt quá giới hạn. Chỉ hoạt động khi bạn đang giới hạn duy nhất một tool; hệ thống sẽ ném ra lỗi `NotImplementedError` nếu có tool khác còn lệnh gọi đang chờ xử lý.

**Ví dụ đầy đủ**

Chỉ định giới hạn với:

* **Thread limit** - Số lệnh gọi tối đa qua tất cả các lần chạy trong một cuộc trò chuyện (yêu cầu checkpointer)
* **Run limit** - Số lệnh gọi tối đa trên mỗi lần kích hoạt agent (reset lại sau mỗi lượt)

Các hành vi thoát:

* `'continue'` (mặc định) - Chặn các lệnh gọi vượt giới hạn bằng thông báo lỗi, agent tiếp tục chạy
* `'error'` - Ném ra ngoại lệ ngay lập tức
* `'end'` - Dừng lại kèm theo ToolMessage + tin nhắn AI (chỉ dùng cho tình huống có một tool duy nhất)

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolCallLimitMiddleware


global_limiter = ToolCallLimitMiddleware(thread_limit=20, run_limit=10)
search_limiter = ToolCallLimitMiddleware(tool_name="search", thread_limit=5, run_limit=3)
database_limiter = ToolCallLimitMiddleware(tool_name="query_database", thread_limit=10)
strict_limiter = ToolCallLimitMiddleware(tool_name="scrape_webpage", run_limit=2, exit_behavior="error")

agent = create_agent(
    model="gpt-5.5",
    tools=[search_tool, database_tool, scraper_tool],
    middleware=[global_limiter, search_limiter, database_limiter, strict_limiter],
)

### PII detection

Phát hiện và xử lý thông tin định danh cá nhân (PII) trong các cuộc trò chuyện bằng các chiến lược có thể cấu hình. Phát hiện PII rất hữu ích cho các trường hợp sau:

* Các ứng dụng tài chính và y tế có yêu cầu tuân thủ bảo mật.
* Các agent chăm sóc khách hàng cần làm sạch nhật ký.
* Bất kỳ ứng dụng nào xử lý dữ liệu nhạy cảm của người dùng.

<div class="alert alert-info">

Khi thiết lập `apply_to_output=True`, `PIIMiddleware` cũng sẽ che giấu đầu ra dạng stream—text deltas, các argument của tool-call, đầu ra của tool và state snapshots - thông qua một stream transformer đã đăng ký. Yêu cầu `langchain>=1.3.2`. Xem thêm [Đăng ký transformer trên middleware](https://docs.langchain.com/oss/python/langchain/event-streaming#register-transformers-on-middleware).

</div>

**Tài liệu API:** [`PIIMiddleware`](https://reference.langchain.com/python/langchain/agents/middleware/pii/PIIMiddleware)

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware

agent = create_agent(
    model="gpt-5.5",
    tools=[],
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
    ],
)

#### Các loại PII tùy chỉnh

Bạn có thể tạo các loại PII tùy chỉnh bằng cách cung cấp một tham số `detector`. Điều này cho phép bạn phát hiện các pattern đặc thù riêng cho use-case của mình ngoài các loại PII được hỗ trợ sẵn.

**Ba cách để tạo detector tùy chỉnh:**

1. **Chuỗi mẫu Regex** - Khớp mẫu đơn giản

2. **Hàm tùy chỉnh** - Logic phát hiện phức tạp kết hợp với việc xác thực

In [ ]:
import re

from langchain.agents import create_agent
from langchain.agents.middleware import PIIMatch, PIIMiddleware


# Cách 1: Chuỗi mẫu Regex
agent1 = create_agent(
    model="gpt-5.5",
    tools=[],
    middleware=[
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
        ),
    ],
)

# Cách 2: Mẫu regex đã được biên dịch
agent2 = create_agent(
    model="gpt-5.5",
    tools=[],
    middleware=[
        PIIMiddleware(
            "phone_number",
            detector=re.compile(r"\+?\d{1,3}[\s.-]?\d{3,4}[\s.-]?\d{4}"),
            strategy="mask",
        ),
    ],
)

# Cách 3: Hàm phát hiện tùy chỉnh
def detect_ssn(content: str) -> list[PIIMatch]:
    """Phát hiện SSN kèm theo xác thực."""
    matches: list[PIIMatch] = []
    pattern = r"\d{3}-\d{2}-\d{4}"
    for match in re.finditer(pattern, content):
        ssn = match.group(0)
        # Xác thực: 3 chữ số đầu tiên không được là 000, 666 hoặc 900-999
        first_three = int(ssn[:3])
        if first_three not in [0, 666] and not (900 <= first_three <= 999):
            matches.append({
                "type": "ssn",
                "value": ssn,
                "start": match.start(),
                "end": match.end(),
            })
    return matches

agent3 = create_agent(
    model="gpt-5.5",
    tools=[],
    middleware=[
        PIIMiddleware(
            "ssn",
            detector=detect_ssn,
            strategy="hash",
        ),
    ],
)

**Chữ ký (signature) của hàm detector tùy chỉnh:**

Hàm detector phải nhận đầu vào là một string (`content`) và trả về các kết quả trùng khớp:

Trả về một danh sách các object `PIIMatch`:

In [ ]:
from langchain.agents.middleware import PIIMatch


def detector(content: str) -> list[PIIMatch]:
    return [
        {
            "type": "custom_type",
            "value": "văn_bản_khớp",
            "start": 0,
            "end": 12,
        },
        # ... các kết quả khớp khác
    ]

<div class="alert alert-success">

Đối với các detector tùy chỉnh:

* Sử dụng chuỗi regex cho các pattern đơn giản.
* Sử dụng đối tượng RegExp khi bạn cần thêm các cờ (ví dụ: khớp không phân biệt chữ hoa chữ thường).
* Sử dụng hàm tùy chỉnh khi bạn cần các logic xác thực bên ngoài việc khớp pattern thông thường.
* Các hàm tùy chỉnh cung cấp toàn quyền kiểm soát đối với logic phát hiện và có thể triển khai các quy tắc xác thực phức tạp.

</div>

**Các tùy chọn cấu hình**

* `pii_type` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">string</span> <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">required</span>
  
  Loại PII cần phát hiện. Có thể là một loại có sẵn (`email`, `credit_card`, `ip`, `mac_address`, `url`) hoặc tên loại PII tùy chỉnh.

* `strategy` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">string</span> <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">default:"redact"</span>
    
  Cách xử lý PII khi phát hiện. Tùy chọn:

  * `'block'` - Ném ra ngoại lệ khi được phát hiện
  * `'redact'` - Thay thế bằng `[REDACTED_{PII_TYPE}]`
  * `'mask'` - Che giấu một phần (ví dụ: `****-****-****-1234`)
  * `'hash'` - Thay thế bằng một mã băm nhất quán

* `detector` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">function | regex</span>
  
  Hàm phát hiện hoặc pattern regex tùy chỉnh. Nếu không được cung cấp, nó sẽ sử dụng bộ phát hiện mặc định cho loại PII tương ứng.

* `apply_to_input` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">boolean</span> <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">default:"True"</span>
  
  Kiểm tra tin nhắn người dùng trước khi gọi model

* `apply_to_output` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">boolean</span> <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">default:"False"</span>

  Kiểm tra tin nhắn AI sau khi model được gọi. Với `langchain>=1.3.2`, tính năng này cũng sẽ che giấu các luồng output dạng stream (text deltas, arguments của tool-call, kết quả của tool, state snapshots) thông qua một stream transformer được đăng ký. Xem [event streaming](https://docs.langchain.com/oss/python/langchain/event-streaming#register-transformers-on-middleware).

* `apply_to_tool_results` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">boolean</span> <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">default:"False"</span>
  
  Kiểm tra các tin nhắn kết quả của tool sau khi thực thi

### To-do list

Trang bị cho các agent khả năng theo dõi và lập kế hoạch cho các tác vụ đa bước phức tạp. To-do list rất hữu ích cho các trường hợp sau:

* Các tác vụ đa bước phức tạp đòi hỏi sự phối hợp giữa nhiều tool.
* Các hoạt động chạy dài hạn nơi mà việc giám sát tiến độ là vô cùng quan trọng.

<div class="alert alert-info">

Middleware này tự động cung cấp cho các agent một tool `write_todos` và các system prompt để hướng dẫn lên kế hoạch thực thi công việc một cách hiệu quả.

</div>

**Tài liệu API:** [`TodoListMiddleware`](https://reference.langchain.com/python/langchain/agents/middleware/todo/TodoListMiddleware)

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import TodoListMiddleware

agent = create_agent(
    model="gpt-5.5",
    tools=[read_file, write_file, run_tests],
    middleware=[TodoListMiddleware()],
)

<div class="alert alert-info">

Xem [video hướng dẫn](https://www.youtube.com/watch?v=yTWocbVKQxw) minh họa hành vi của middleware To-do List.

</div>

**Các tùy chọn cấu hình**

* `system_prompt` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">string</span>
  
  System prompt tùy chỉnh để hướng dẫn sử dụng todo. Nếu không được cung cấp, prompt mặc định sẽ được dùng.

* `tool_description` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">string</span>
  
  Mô tả tùy chỉnh cho tool `write_todos`. Nếu không được cung cấp, mô tả mặc định sẽ được dùng.

### LLM tool selector

Sử dụng một LLM để chọn ra một cách thông minh các tool liên quan nhất trước khi gọi model chính. Bộ chọn tool LLM rất hữu ích cho các trường hợp sau:

* Các agent sở hữu quá nhiều tool (10+) mà đa phần không liên quan đối với từng truy vấn cụ thể.
* Giảm mức sử dụng token bằng cách lọc bớt các tool không cần thiết.
* Cải thiện sự tập trung và độ chính xác của model.

Middleware này sử dụng structured output để yêu cầu một LLM xác định xem những tool nào phù hợp nhất với truy vấn hiện tại. Schema của structured output định nghĩa các tên và mô tả của các tool khả dụng. Đằng sau hậu trường, các provider model thường bổ sung thông tin structured output này vào system prompt.

**Tài liệu API:** [`LLMToolSelectorMiddleware`](https://reference.langchain.com/python/langchain/agents/middleware/tool_selection/LLMToolSelectorMiddleware)

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import LLMToolSelectorMiddleware

agent = create_agent(
    model="gpt-5.5",
    tools=[tool1, tool2, tool3, tool4, tool5, ...],
    middleware=[
        LLMToolSelectorMiddleware(
            model="gpt-5.4-mini",
            max_tools=3,
            always_include=["search"],
        ),
    ],
)

**Các tùy chọn cấu hình**

* `model` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">string | BaseChatModel</span>

  Model dùng cho việc lựa chọn tool. Có thể là một chuỗi định danh model (ví dụ: `'openai:gpt-5.4-mini'`) hoặc một instance của `BaseChatModel`. Xem [`init_chat_model`](https://reference.langchain.com/python/langchain/chat_models/base/init_chat_model) để biết thêm thông tin.

  Mặc định sử dụng model chính của agent.

* `system_prompt` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">string</span>

  Hướng dẫn dành cho model chọn tool. Sử dụng prompt mặc định nếu không được chỉ định.

* `max_tools` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">number</span>

  Số lượng tool tối đa để chọn. Nếu model chọn nhiều hơn, chỉ tối đa `max_tools` công cụ đầu tiên sẽ được sử dụng. Nếu không được chỉ định thì mặc định không có giới hạn.

* `always_include` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">list[string]</span>

  Tên của các tool luôn luôn được bao gồm bất chấp kết quả lựa chọn từ LLM. Các tool này không được tính vào giới hạn `max_tools`.

### Provider tool search

Trì hoãn các tool đã chọn đằng sau tính năng tìm kiếm tool ở phía server của provider model, do đó model sẽ khám phá chúng theo nhu cầu thay vì nhận trước toàn bộ schema của tất cả các tool. Tính năng tìm kiếm tool của provider cực kỳ hữu ích cho:

* Giảm việc context bị phình to khi sử dụng số lượng lớn tool.
* Cải thiện độ chính xác trong việc lựa chọn tool thông qua việc chỉ hiển thị các tool thực sự liên quan.

<div class="alert alert-info">

Yêu cầu model phải có hỗ trợ tính năng tìm kiếm tool trên server: Anthropic (Claude Sonnet 4+/Opus 4+/Haiku 4.5+) hoặc OpenAI (gpt-5.5+). Các provider khác sẽ ném ra lỗi `ValueError`.

</div>

**Tài liệu API:** [`ProviderToolSearchMiddleware`](https://reference.langchain.com/python/langchain/agents/middleware/provider_tool_search/ProviderToolSearchMiddleware)

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ProviderToolSearchMiddleware

agent = create_agent(
    model="anthropic:claude-opus-4-8",
    tools=[get_weather, lookup_order],
    middleware=[
        ProviderToolSearchMiddleware(searchable_tools=["lookup_order"]),
    ],
)

**Các tùy chọn cấu hình**

* `searchable_tools` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">list[str | BaseTool]</span>

  Các tool sẽ được đưa ra phía sau tính năng tìm kiếm của provider (bằng tên gọi hoặc instance). Các tool bị trì hoãn này sẽ được giữ lại, không hiển thị cho model cho tới khi kết quả tìm kiếm của nó truy xuất đến. Các tool được khởi tạo với `extras={"defer_loading": True}` sẽ luôn bị trì hoãn bất kể tùy chọn này ra sao; nếu bỏ qua thông số `searchable_tools`, thì chỉ các tool đã được đánh dấu sẵn đó mới bị trì hoãn.

**Ví dụ đầy đủ**

Middleware này áp dụng tính năng trì hoãn và tìm kiếm đối với toàn bộ tool được liệt kê trong `searchable_tools`. Một tool cũng có thể được cài đặt để trì hoãn ngay từ khi khởi tạo bằng cách cấu hình `extras={"defer_loading": True}`.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ProviderToolSearchMiddleware
from langchain.tools import tool


# Đã được đánh dấu defer_loading khi khởi tạo —
# nên nó tự động được trì hoãn (deferred) — không cần liệt kê trong searchable_tools.
@tool(extras={"defer_loading": True})
def send_email(to: str) -> str:
    """Gửi email."""
    return "đã gửi"


agent = create_agent(
    model="anthropic:claude-opus-4-8",
    tools=[send_email],
    middleware=[ProviderToolSearchMiddleware()],
)

### Shell tool

Cung cấp một phiên làm việc shell bền bỉ cho các agent để thực thi các lệnh hệ thống. Middleware Shell tool đặc biệt hữu ích cho các trường hợp:

* Agent cần thực thi các lệnh trên hệ thống.
* Tự động hóa các tác vụ phát triển và triển khai.
* Các quy trình kiểm thử và xác thực.
* Các thao tác với file system và chạy script.

<div class="alert alert-warning">

**Lưu ý bảo mật**: Hãy sử dụng các chính sách thực thi phù hợp (`HostExecutionPolicy`, `DockerExecutionPolicy`, hoặc `CodexSandboxExecutionPolicy`) để khớp với các yêu cầu về bảo mật trên môi trường triển khai của bạn.

</div>

<div class="alert alert-info">

**Hạn chế**: Hiện tại, các phiên shell duy trì không hoạt động với các thao tác gián đoạn (chẳng hạn như human-in-the-loop). Chúng tôi dự kiến sẽ bổ sung hỗ trợ tính năng này trong tương lai.

</div>

**Tài liệu API:** [`ShellToolMiddleware`](https://reference.langchain.com/python/langchain/agents/middleware/shell_tool/ShellToolMiddleware)

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import (
    ShellToolMiddleware,
    HostExecutionPolicy,
)

agent = create_agent(
    model="gpt-5.5",
    tools=[search_tool],
    middleware=[
        ShellToolMiddleware(
            workspace_root="/workspace",
            execution_policy=HostExecutionPolicy(),
        ),
    ],
)

**Các tùy chọn cấu hình**

* `workspace_root` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">str | Path | None</span>

  Thư mục gốc cho phiên làm việc shell. Nếu để trống, một thư mục tạm thời sẽ được tạo khi agent khởi động và bị xóa khi kết thúc.

* `startup_commands` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">tuple[str, ...] | list[str] | str | None</span>

  Các lệnh tùy chọn sẽ được chạy theo tuần tự sau khi phiên shell khởi động.

* `shutdown_commands` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">tuple[str, ...] | list[str] | str | None</span>

  Các lệnh tùy chọn sẽ được chạy trước khi phiên shell kết thúc.

* `execution_policy` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">BaseExecutionPolicy | None</span>

  Chính sách thực thi cho phép kiểm soát timeout, giới hạn đầu ra, và cấu hình tài nguyên. Tùy chọn:

  * `HostExecutionPolicy` - Quyền truy cập đầy đủ vào máy chủ (mặc định); tốt nhất cho các môi trường an toàn khi mà bản thân agent đã được vận hành sẵn trong một container hoặc máy ảo (VM)
  * `DockerExecutionPolicy` - Khởi tạo một container Docker riêng cho từng lần chạy của agent, mang lại mức độ cách ly nghiêm ngặt hơn
  * `CodexSandboxExecutionPolicy` - Tái sử dụng hệ thống sandbox của Codex CLI nhằm giới hạn thêm các hàm syscall/filesystem

* `redaction_rules` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">tuple[RedactionRule, ...] | list[RedactionRule] | None</span>

  Tùy chọn quy tắc che giấu/chỉnh sửa để làm sạch kết quả lệnh trước khi gửi về cho model.

  <div class="alert alert-warning">

  Quy tắc che giấu/chỉnh sửa chỉ được áp dụng sau khi lệnh được thực thi và không có tác dụng ngăn chặn các dữ liệu nhạy cảm rò rỉ ra ngoài khi dùng `HostExecutionPolicy`.
  
  </div>

* `tool_description` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">str | None</span>

  Cho phép tùy chọn ghi đè cấu hình mô tả của shell tool đã được đăng ký

* `shell_command` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">Sequence[str] | str | None</span>

  Đường dẫn lệnh khởi động shell (dạng string) hoặc chuỗi argument tùy chọn để tạo ra phiên session bền bỉ. Mặc định là `/bin/bash`.

* `env` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">Mapping[str, Any] | None</span>

  Các biến môi trường tùy chọn sẽ cấp cho shell session. Mọi giá trị sẽ được ép kiểu về dạng chuỗi (string) trước khi lệnh được chạy.

**Ví dụ đầy đủ**

Middleware này cung cấp một phiên làm việc shell bền bỉ đơn nhất giúp agent có thể thực thi các lệnh tuần tự.

**Các chính sách thực thi:**

* `HostExecutionPolicy` (mặc định) - Thực thi gốc với đầy đủ quyền truy cập hệ thống máy chủ.
* `DockerExecutionPolicy` - Thực thi cách ly bằng một container Docker.
* `CodexSandboxExecutionPolicy` - Thực thi ở môi trường sandbox thông qua Codex CLI.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import (
    ShellToolMiddleware,
    HostExecutionPolicy,
    DockerExecutionPolicy,
    RedactionRule,
)


# Shell tool cơ bản thực thi trên máy chủ (host)
agent = create_agent(
    model="gpt-5.5",
    tools=[search_tool],
    middleware=[
        ShellToolMiddleware(
            workspace_root="/workspace",
            execution_policy=HostExecutionPolicy(),
        ),
    ],
)

# Cách ly bằng Docker với các lệnh khởi động
agent_docker = create_agent(
    model="gpt-5.5",
    tools=[],
    middleware=[
        ShellToolMiddleware(
            workspace_root="/workspace",
            startup_commands=["pip install requests", "export PYTHONPATH=/workspace"],
            execution_policy=DockerExecutionPolicy(
                image="python:3.11-slim",
                command_timeout=60.0,
            ),
        ),
    ],
)

# Chỉnh sửa/che giấu đầu ra (áp dụng sau khi thực thi)
agent_redacted = create_agent(
    model="gpt-5.5",
    tools=[],
    middleware=[
        ShellToolMiddleware(
            workspace_root="/workspace",
            redaction_rules=[
                RedactionRule(pii_type="api_key", detector=r"sk-[a-zA-Z0-9]{32}"),
            ],
        ),
    ],
)

### Filesystem

Context engineering là một trong những thử thách chính trong việc thiết kế agent hiệu quả. Điều này đặc biệt nan giải đối với các tool trả về nội dung với kích thước có thể thay đổi (ví dụ: `web_search` và RAG), vì chỉ cần một kết quả tool dài thôi cũng đủ để làm tràn context window.

`FilesystemMiddleware` của [Deep Agents](https://docs.langchain.com/oss/python/deepagents/overview) mang đến 4 công cụ tương tác cho cả bộ nhớ dài hạn và bộ nhớ ngắn hạn:

* `ls`: Liệt kê các tệp có trong filesystem
* `read_file`: Đọc nội dung toàn bộ tệp hoặc đọc một số dòng cụ thể từ tệp
* `write_file`: Viết nội dung mới vào một tệp trên filesystem
* `edit_file`: Thay đổi nội dung của một tệp hiện có trên filesystem

In [ ]:
from langchain.agents import create_agent
from deepagents.middleware.filesystem import FilesystemMiddleware

# FilesystemMiddleware được bao gồm mặc định trong create_deep_agent
# Bạn có thể tùy chỉnh nó nếu xây dựng một agent tùy chỉnh
agent = create_agent(
    model="claude-sonnet-4-6",
    middleware=[
        FilesystemMiddleware(
            backend=None,  # Tùy chọn: backend tùy chỉnh (mặc định là StateBackend)
            system_prompt="Ghi vào hệ thống tệp khi...",  # Tùy chọn: Thêm nội dung tùy chỉnh vào system prompt
            custom_tool_descriptions={
                "ls": "Sử dụng tool ls khi...",
                "read_file": "Sử dụng tool read_file để..."
            },  # Tùy chọn: Các mô tả tùy chỉnh cho các tool của filesystem
            tools=["read_file", "ls", "glob", "grep"],  # Tùy chọn: Allowlist giới hạn các tool filesystem nào được hiển thị
        ),
    ],
)

#### Short-term so với long-term

Mặc định, những công cụ này sẽ ghi kết quả vào một filesystem cục bộ trên graph state của bạn. Để kích hoạt khả năng lưu trữ liên tục thông qua các thread, hãy cấu hình `CompositeBackend` dùng để định tuyến đường dẫn cụ thể (ví dụ `/memories/`) tới `StoreBackend`.

In [ ]:
from langchain.agents import create_agent
from deepagents.middleware import FilesystemMiddleware
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

agent = create_agent(
    model="claude-sonnet-4-6",
    store=store,
    middleware=[
        FilesystemMiddleware(
            backend=CompositeBackend(
                default=StateBackend(),
                routes={"/memories/": StoreBackend()}
            ),
            custom_tool_descriptions={
                "ls": "Sử dụng tool ls khi...",
                "read_file": "Sử dụng tool read_file để..."
            }  # Tùy chọn: Các mô tả tùy chỉnh cho các tool của filesystem
        ),
    ],
)

Khi cấu hình một `CompositeBackend` kèm với `StoreBackend` ứng với thư mục `/memories/`, mọi tệp mang tiền tố **/memories/** sẽ tự động được ghi lại trên cơ sở lưu trữ vĩnh viễn và tồn tại an toàn xuyên suốt các thread khác nhau. Các tệp không nằm trong đường dẫn đó sẽ tiếp tục được giữ lại trên ephemeral state storage.

### Subagent

Chuyển giao các tác vụ cho subagent cho phép cô lập context, qua đó giúp cho context window của main agent luôn sạch gọn trong khi vẫn đảm bảo mức độ chuyên sâu của tác vụ.

Middleware subagent của [Deep Agents](https://docs.langchain.com/oss/python/deepagents/overview) hỗ trợ việc phân quyền cho các subagent thông qua một tool `task`.

In [ ]:
from langchain.tools import tool
from langchain.agents import create_agent
from deepagents.middleware.subagents import SubAgentMiddleware


@tool
def get_weather(city: str) -> str:
    """Lấy thông tin thời tiết của một thành phố."""
    return f"Thời tiết ở {city} đang có nắng."

agent = create_agent(
    model="claude-sonnet-4-6",
    middleware=[
        SubAgentMiddleware(
            default_model="claude-sonnet-4-6",
            default_tools=[],
            subagents=[
                {
                    "name": "weather",
                    "description": "Subagent này có thể lấy thông tin thời tiết ở các thành phố.",
                    "system_prompt": "Sử dụng tool get_weather để lấy thông tin thời tiết của một thành phố.",
                    "tools": [get_weather],
                    "model": "gpt-5.5",
                    "middleware": [],
                }
            ],
        )
    ],
)

Một subagent được định nghĩa bằng **name**, **description**, **system prompt** và các **tool**. Bạn có thể tùy chọn gán một **model** riêng hoặc thêm các **middleware** cụ thể vào subagent đó. Điều này tỏ ra đặc biệt hữu hiệu khi bạn muốn cung cấp thêm key trạng thái đặc biệt cho subagent để dùng chung với agent chính.

Dành cho những trường hợp phức tạp hơn, bạn hoàn toàn có thể tự đưa vào một graph LangGraph đã được cấu hình sẵn dưới hình thức một subagent.

In [ ]:
from langchain.agents import create_agent
from deepagents.middleware.subagents import SubAgentMiddleware
from deepagents import CompiledSubAgent
from langgraph.graph import StateGraph

# Tạo một LangGraph graph tùy chỉnh
def create_weather_graph():
    workflow = StateGraph(...)
    # Xây dựng graph tùy chỉnh của bạn
    return workflow.compile()

weather_graph = create_weather_graph()

# Bọc nó trong một CompiledSubAgent
weather_subagent = CompiledSubAgent(
    name="weather",
    description="Subagent này có thể lấy thông tin thời tiết ở các thành phố.",
    runnable=weather_graph
)

agent = create_agent(
    model="claude-sonnet-4-6",
    middleware=[
        SubAgentMiddleware(
            default_model="claude-sonnet-4-6",
            default_tools=[],
            subagents=[weather_subagent],
        )
    ],
)

Ngoài các subagent do người dùng tự xây dựng, main agent mặc định sẽ có quyền điều khiển một subagent `general-purpose` (đa dụng) trong mọi thời điểm. Subagent này có các instruction cùng tất cả các quyền truy cập tool tương đương với main agent. Chức năng chính của `general-purpose` subagent là để chia tách context - main agent có thể giao một tác vụ dài phức tạp qua cho subagent đó và chỉ nhận về một câu trả lời ngắn gọn, ngăn chặn tình trạng phình to do hệ quả của những đợt gọi tool.

### Rubric grading

<div class="alert alert-info">

`RubricMiddleware` yêu cầu phiên bản `deepagents>=0.6.5`. Chức năng này hiện đang ở giai đoạn [**beta**](https://docs.langchain.com/oss/python/versioning); cấu trúc API có thể sẽ thay đổi trong tương lai.

</div>

Một số tác vụ có yêu cầu kết quả cuối cùng rất cụ thể ("done") mà agent khó lòng đạt được ngay trong lần thử nghiệm đầu tiên. `RubricMiddleware` cho phép bạn đưa ra bộ tiêu chuẩn đánh giá kết quả mong muốn và thiết lập cho agent khả năng tự kiểm định chất lượng rồi chỉnh sửa kết quả cho đến khi thỏa mãn hoàn toàn bộ yêu cầu, hoặc cho đến khi vượt quá số lần chỉnh sửa tối đa.

**Tài liệu API:** [`RubricMiddleware`](https://reference.langchain.com/python/deepagents/middleware/rubric/RubricMiddleware)

In [ ]:
from deepagents import RubricMiddleware, create_deep_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_deep_agent(
    model="openai:gpt-5.5",
    middleware=[
        RubricMiddleware(
            model="anthropic:claude-haiku-4-5",
            max_iterations=3,
        ),
    ],
    checkpointer=InMemorySaver(),
)

Để nắm được các tùy chọn cấu hình đầy đủ, xử lý luồng stream và một kịch bản sinh mã chi tiết, mời tham khảo mục [Grading rubrics](https://docs.langchain.com/oss/python/deepagents/rubric).

### File search

Cung cấp công cụ Glob search và Grep search hoạt động trên môi trường file system. Middleware tìm kiếm tệp mang lại hiệu suất tốt cho các ứng dụng như:

* Kiểm tra và phân tích source code.
* Tìm kiếm tệp thông qua các pattern trong tên tệp.
* Tìm kiếm nội dung bên trong source code dựa trên các pattern regex.
* Codebase lớn với nhu cầu tìm và truy xuất nhiều tệp tin.

**Tài liệu API:** [`FilesystemFileSearchMiddleware`](https://reference.langchain.com/python/langchain/agents/middleware/file_search/FilesystemFileSearchMiddleware)

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import FilesystemFileSearchMiddleware

agent = create_agent(
    model="gpt-5.5",
    tools=[],
    middleware=[
        FilesystemFileSearchMiddleware(
            root_path="/workspace",
            use_ripgrep=True,
        ),
    ],
)

**Các tùy chọn cấu hình**

* `root_path` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">str</span> <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">required</span>
  
  Đường dẫn của thư mục gốc để bắt đầu tìm kiếm. Tất cả mọi thao tác liên quan tới tệp tin đều mang tính chất tương đối và dựa trên đường dẫn gốc này.

* `use_ripgrep` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">bool</span> <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">default:"True"</span>

  Tùy chọn xác định xem có nên sử dụng ripgrep cho các truy vấn. Sẽ chuyển qua sử dụng Python regex mặc định (fallback) trong trường hợp ripgrep không khả dụng.

* `max_file_size_mb` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">int</span> <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">default:"10"</span>

  Kích thước tối đa của tệp tính bằng MB mà công cụ tìm kiếm hoạt động. Sẽ bỏ qua và không tìm kiếm nội dung các tệp vượt giới hạn này.


**Ví dụ đầy đủ**

Middleware bổ sung hai tool tìm kiếm thông dụng vào các agent:

**Glob tool** - Hỗ trợ thao tác kết nối mẫu đối với tên file siêu tốc:

* Chấp nhận hàng loạt các pattern như `**/*.py`, `src/**/*.ts`
* Trả lại các tập tin có kết quả trùng khớp theo thứ tự ưu tiên chỉnh sửa gần nhất

**Grep tool** - Dùng pattern regex tìm nội dung:

* Khai thác toàn bộ cú pháp từ thư viện regex
* Kết hợp pattern tìm file từ lựa chọn của tham số `include`
* Hỗ trợ ba dạng định dạng trả về bao gồm: `files_with_matches`, `content`, và `count`

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import FilesystemFileSearchMiddleware
from langchain.messages import HumanMessage


agent = create_agent(
    model="gpt-5.5",
    tools=[],
    middleware=[
        FilesystemFileSearchMiddleware(
            root_path="/workspace",
            use_ripgrep=True,
            max_file_size_mb=10,
        ),
    ],
)

# Agent hiện có thể sử dụng các tool glob_search và grep_search
result = agent.invoke({
    "messages": [HumanMessage("Tìm tất cả các tệp Python có chứa 'async def'")]
})

# Agent sẽ sử dụng:
# 1. glob_search(pattern="**/*.py") để tìm các tệp Python
# 2. grep_search(pattern="async def", include="*.py") để tìm các hàm async

### Context editing

Quản lý context hội thoại qua việc làm sạch kết quả của những lệnh gọi tool phiên bản cũ trước đó khi sắp bị quá tải giới hạn của model, nhưng bảo đảm duy trì kết quả gần nhất trong context. Giúp xử lý và điều tiết hiệu quả context window đối với các kịch bản đối thoại quy mô lớn và sở hữu số lượng tool call đồ sộ. Chỉnh sửa context tỏ ra cực kỳ hiệu quả đối với các trường hợp:

* Các hội thoại dài kèm theo khối lượng lệnh gọi tool lớn và có nguy cơ đụng hạn mức giới hạn token
* Giảm hao phí chi phí qua cách loại trừ các lệnh gọi tool thế hệ cũ không còn tác dụng vào bối cảnh truy vấn hiện tại.
* Cho phép ghim (duy trì) cố định một số N kết quả của các tool call vào trong context.

**Tài liệu API:** [`ContextEditingMiddleware`](https://reference.langchain.com/python/langchain/agents/middleware/context_editing/ContextEditingMiddleware), [`ClearToolUsesEdit`](https://reference.langchain.com/python/langchain/agents/middleware/context_editing/ClearToolUsesEdit)

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ContextEditingMiddleware, ClearToolUsesEdit

agent = create_agent(
    model="gpt-5.5",
    tools=[],
    middleware=[
        ContextEditingMiddleware(
            edits=[
                ClearToolUsesEdit(
                    trigger=100000,
                    keep=3,
                ),
            ],
        ),
    ],
)

**Các tùy chọn cấu hình**

* `edits` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">list[ContextEdit]</span> <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">default:"[ClearToolUsesEdit()]"</span>

  Danh sách các strategy [`ContextEdit`](https://reference.langchain.com/python/langchain/agents/middleware/context_editing/ContextEdit) để áp dụng.

* `token_count_method` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">string</span> <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">default:"approximate"</span>

  Phương pháp tính (đếm) số token. Tùy chọn: `'approximate'` (ương chừng/gần đúng) hoặc `'model'`.

  **Các tùy chọn của [`ClearToolUsesEdit`](https://reference.langchain.com/python/langchain/agents/middleware/context_editing/ClearToolUsesEdit):**

* `trigger` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">number</span> <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">default:"100000"</span>

  Hạn mức token để kích hoạt chỉnh sửa. Bất cứ khi nào chuỗi giao tiếp hội thoại đụng ngưỡng lượng token này, kết quả của các lệnh tool ở phiên trước tự động được làm sạch.
  
* `clear_at_least` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">number</span> <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">default:"0"</span>

  Khối lượng token nhỏ nhất muốn bớt ra ở phiên biên tập (edit). Nếu đặt giá trị 0, sẽ tự động xóa cho đến khi vừa đủ nhu cầu xử lý.

* `keep` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">number</span> <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">default:"3"</span>

  Chỉ định giá trị số lượng kết quả của các tool call thuộc thế hệ gần nhất buộc phải giữ nguyên trạng thái không sửa đổi. Những nội dung được chỉ định sẽ vĩnh viễn không bị xóa nhầm.

* `clear_tool_inputs` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">boolean</span> <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">default:"False"</span>

  Dùng cấu hình này trong trường hợp bạn có mong muốn làm trống cả giá trị biến đổi của tham số đối với đoạn nhắn tin từ AI. Nếu giá trị chuyển `True`, các thông số argument sẽ biến đổi sang chuỗi empty objects.

* `exclude_tools` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">list[string]</span> <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">default:"()"</span>

  Lựa chọn tên của các list chứa nhiều tool đặc biệt nhằm vô hiệu hóa quy luật làm sạch. Kết quả trả về qua những lệnh tool trên thì sẽ vĩnh viễn được nguyên hiện trạng.

* `placeholder` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">string</span> <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">default:"[cleared]"</span>

  Chèn chuỗi ký tự placeholder thế chỗ tại khu vực xóa bỏ các kết quả của phiên trả lời từ tool trước. Mẫu đoạn placeholder này thay thế hoàn toàn chuỗi văn bản tin nhắn.

**Ví dụ đầy đủ**

Middleware áp dụng các phương pháp can thiệp khi hội thoại báo động quá tải hạn ngạch lượng ký tự token. Biện pháp khả dụng nhất thuộc loại `ClearToolUsesEdit`, mang trách nhiệm dọn dẹp các lịch sử lệnh tool qua một chu kỳ nhất định nhằm tiết kiệm cho các tương tác hội thoại gần sát thực tại.

**Cách thức hoạt động:**

1. Kiểm soát theo dõi tổng token trong luồng hội thoại
2. Bất cứ khi nào tín hiệu giới hạn (threshold) báo đỏ (bị vi phạm), sẽ tiến hành xoá output của công cụ thế hệ đã cũ
3. Chỉ cho phép các lệnh lịch sử N đợt (kết quả) gần đây không ảnh hưởng
4. Lựa chọn bảo vệ cấu trúc chuỗi của argument gốc cho đoạn truy xuất

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ContextEditingMiddleware, ClearToolUsesEdit


agent = create_agent(
    model="gpt-5.5",
    tools=[search_tool, your_calculator_tool, database_tool],
    middleware=[
        ContextEditingMiddleware(
            edits=[
                ClearToolUsesEdit(
                    trigger=2000,
                    keep=3,
                    clear_tool_inputs=False,
                    exclude_tools=[],
                    placeholder="[đã xóa]",
                ),
            ],
        ),
    ],
)

### LLM tool emulator

Dùng LLM mô phỏng (giả lập) lệnh tool dành riêng đối với nhiệm vụ xác thực thực nghiệm (testing), nhằm thế chỗ hoàn toàn các lệnh thực ở tool thành hệ thống mã phản hồi phỏng đoán theo cách sinh học qua AI. Các trình mô phỏng LLM mang ưu điểm mạnh trên các mục đích sau:

* Kiểm nghiệm các ứng xử qua tương tác agent hoàn toàn miễn là không buộc chạy bất kể tool hệ thống.
* Trợ giúp thiết lập các mạng lưới agent ngay cả khi đang vướng bận kết nối ngoài mạng (hoặc khi phí ngoài API tốn kém đắt đỏ).
* Hình thành (prototype) quy trình tự động hóa thao tác khi mà hệ thống ứng dụng thật chưa ra mắt thành hình hài đầy đủ.

**Tài liệu API:** [`LLMToolEmulator`](https://reference.langchain.com/python/langchain/agents/middleware/tool_emulator/LLMToolEmulator)

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import LLMToolEmulator

agent = create_agent(
    model="gpt-5.5",
    tools=[get_weather, search_database, send_email],
    middleware=[
        LLMToolEmulator(),  # Giả lập tất cả các tool
    ],
)

**Các tùy chọn cấu hình**

* `tools` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">list[str | BaseTool]</span>

  Danh sách xác định danh xưng tên gọi (str) hoăc class instance `BaseTool` muốn đem thử nghiệm qua (giả lập). Đặt theo tuỳ mặc định `None` -> giả lập 100% tập danh mục tool trong bảng danh sách đã đăng ký. Để ở list không `[]` -> loại bỏ hoàn toàn tiến trình qua lệnh mô phỏng. Khi list nhận dạng có khai rõ class instance, thì chỉ danh mục có mặt trên tên gọi đó tham gia vào lệnh giả lập.

* `model` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">string | BaseChatModel</span>

  Gán model chuyên trách chịu trọng trách cho luồng mô phỏng quá trình tạo phản hồi mô phỏng. Cho phép sử dụng string định vị model (ví dụ `'google_genai:gemini-3.6-flash'`) hoặc qua trực tiếp trên instance `BaseChatModel`. Tự động gọi mặc định theo class hệ thống đang nắm giữ nhiệm vụ nếu không được cấu hình. Xem [`init_chat_model`](https://reference.langchain.com/python/langchain/chat_models/base/init_chat_model) để biết thêm thông tin.

**Ví dụ đầy đủ**

Middleware sử dụng một LLM để tạo ra các phản hồi hợp lý cho những lệnh gọi tool thay vì tiến hành chạy các tool ngoài đời thực.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import LLMToolEmulator
from langchain.tools import tool


@tool
def get_weather(location: str) -> str:
    """Lấy thông tin thời tiết hiện tại của một địa điểm."""
    return f"Thời tiết ở {location}"

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Gửi email."""
    return "Đã gửi email"


# Giả lập tất cả các tool (hành vi mặc định)
agent = create_agent(
    model="gpt-5.5",
    tools=[get_weather, send_email],
    middleware=[LLMToolEmulator()],
)

# Chỉ giả lập các tool cụ thể
agent2 = create_agent(
    model="gpt-5.5",
    tools=[get_weather, send_email],
    middleware=[LLMToolEmulator(tools=["get_weather"])],
)

# Sử dụng model tùy chỉnh cho việc giả lập
agent4 = create_agent(
    model="gpt-5.5",
    tools=[get_weather, send_email],
    middleware=[LLMToolEmulator(model="claude-sonnet-4-6")],
)

## Middleware dành riêng cho Provider

Các middleware này được tối ưu hóa cho các provider LLM cụ thể. Xem tài liệu của từng provider để biết đầy đủ chi tiết và các ví dụ.

* [**Anthropic**](https://docs.langchain.com/oss/python/integrations/middleware/anthropic): Tính năng cache prompt, bash tool, text editor, memory và middleware tìm kiếm tệp dành riêng cho các dòng model của Claude.
* [**AWS**](https://docs.langchain.com/oss/python/integrations/middleware/aws): Middleware hỗ trợ cache prompt danh riêng cho Amazon Bedrock.
  </Card>
* [**OpenAI**](https://docs.langchain.com/oss/python/integrations/middleware/openai): Middleware chuyên kiểm soát cho dòng sản phẩm tới từ hệ sinh thái OpenAI.